# UVH-26 → YOLOv8n Fine-Tuning (Nexiora Phase 2a)

Run this entire notebook in **Google Colab** with a free GPU runtime
(Runtime → Change runtime type → T4 GPU). Do **not** run this on the
project laptop — no local GPU, see `CLAUDE.md` section 8.

Produces a fine-tuned `best.pt` (4 classes: car / bike / bus / truck,
merged down from UVH-26's 14 fine-grained classes) to replace stock
`yolov8n.pt` in `backend/config.py`.

**Before running the conversion cell unattended**: inspect the printed
sample record and the drawn bounding box a few cells down. If the box
doesn't tightly fit the vehicle, flip `BBOX_FORMAT` (or the field names)
and re-check before converting the full ~26k-image dataset.

In [ ]:
!pip install -q -U ultralytics huggingface_hub tqdm

In [ ]:
from huggingface_hub import login

login()  # paste a read-access token for Vikas-mandlik/UVH-26-bucket when prompted

In [ ]:
from pathlib import Path

# Buckets use their own hf://buckets/... URI and the `hf sync` CLI, not the
# regular snapshot_download() dataset API (confirmed via this bucket's
# "Sync this bucket" dialog). `login()` above already wrote the auth token
# that `hf` reads, so no separate CLI login step is needed.
!hf sync hf://buckets/Vikas-mandlik/UVH-26-bucket ./uvh26_data

DATASET_DIR = Path("uvh26_data").resolve()
print(DATASET_DIR)
!ls "$DATASET_DIR"

In [ ]:
import json
from collections import defaultdict

# Confirmed against the actual downloaded JSON (not just the dataset
# card): this is plain COCO format - top-level images/annotations/
# categories lists, bbox is [x, y, width, height] absolute pixels
# (confirmed via area == width*height on real records), category ids
# are 1-14 with "Others" as the catch-all (not "Other").
train_json_path = DATASET_DIR / "UVH-26-Train" / "UVH-26-MV-Train.json"
with open(train_json_path, encoding="utf-8") as f:
    train_json = json.load(f)

images_by_id = {img["id"]: img for img in train_json["images"]}
anns_by_image = defaultdict(list)
for ann in train_json["annotations"]:
    anns_by_image[ann["image_id"]].append(ann)
cat_names = {c["id"]: c["name"] for c in train_json["categories"]}

print(f"{len(images_by_id)} images, {len(train_json['annotations'])} annotations")
print("categories:", cat_names)

sample_img_id = next(iter(images_by_id))
print("sample image record:", images_by_id[sample_img_id])
print("sample annotations:", anns_by_image[sample_img_id][:3])

Confirmed above: this is plain COCO format. Next cell draws the boxes on a real image to double check the `bbox` convention actually matches `[x, y, width, height]` absolute pixels before converting the full dataset.

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches


def build_image_index(images_dir):
    # Images are sharded under images_dir/data/000/, /001/, etc, not flat.
    data_dir = images_dir / "data"
    search_root = data_dir if data_dir.is_dir() else images_dir
    return {p.name: p for p in search_root.rglob("*") if p.is_file()}


train_file_index = build_image_index(DATASET_DIR / "UVH-26-Train")
print(f"found {len(train_file_index)} train image files on disk")

img_rec = images_by_id[sample_img_id]
anns = anns_by_image[sample_img_id]

img_path = train_file_index[img_rec["file_name"]]
img = Image.open(img_path)

fig, ax = plt.subplots(1, figsize=(12, 8))
ax.imshow(img)
for ann in anns:
    x, y, w, h = ann["bbox"]
    cls_name = cat_names[ann["category_id"]]
    rect = patches.Rectangle((x, y), w, h, linewidth=2, edgecolor="lime", facecolor="none")
    ax.add_patch(rect)
    ax.text(x, max(y - 4, 0), cls_name, color="lime", fontsize=9)
plt.show()

## Convert to YOLO format (14 UVH-26 classes → 4 project classes)

In [ ]:
import shutil
from tqdm import tqdm

CLASS_MAP = {
    "Hatchback": "car", "Sedan": "car", "SUV": "car", "MUV": "car", "Van": "car", "Three-wheeler": "car",
    "Bus": "bus", "Mini-bus": "bus",
    "Truck": "truck", "LCV": "truck", "Tempo-traveller": "truck",
    "Two-wheeler": "bike", "Bicycle": "bike",
    "Others": None,  # dropped — not a vehicle class the project tracks
}
TARGET_CLASSES = ["car", "bike", "bus", "truck"]  # final ids 0, 1, 2, 3
TARGET_ID = {name: i for i, name in enumerate(TARGET_CLASSES)}

OUT_ROOT = Path("/content/uvh26_yolo")
for split in ("train", "val"):
    (OUT_ROOT / "images" / split).mkdir(parents=True, exist_ok=True)
    (OUT_ROOT / "labels" / split).mkdir(parents=True, exist_ok=True)


def load_coco(json_path):
    with open(json_path, encoding="utf-8") as f:
        data = json.load(f)
    images_by_id = {img["id"]: img for img in data["images"]}
    anns_by_image = defaultdict(list)
    for ann in data["annotations"]:
        anns_by_image[ann["image_id"]].append(ann)
    cat_names = {c["id"]: c["name"] for c in data["categories"]}
    return images_by_id, anns_by_image, cat_names


def convert_split(split, json_path, images_dir):
    images_by_id, anns_by_image, cat_names = load_coco(json_path)
    file_index = build_image_index(images_dir)
    kept = 0
    for img_id, img_rec in tqdm(images_by_id.items(), desc=split):
        src_img = file_index.get(img_rec["file_name"])
        if src_img is None or not src_img.exists():
            continue
        img_w, img_h = img_rec["width"], img_rec["height"]

        lines = []
        for ann in anns_by_image.get(img_id, []):
            cls_name = cat_names[ann["category_id"]]
            target = CLASS_MAP.get(cls_name)
            if target is None:
                continue
            x, y, w, h = ann["bbox"]
            x1, x2 = sorted((max(0, x), min(img_w, x + w)))
            y1, y2 = sorted((max(0, y), min(img_h, y + h)))
            if x2 <= x1 or y2 <= y1:
                continue
            xc, yc = ((x1 + x2) / 2) / img_w, ((y1 + y2) / 2) / img_h
            bw, bh = (x2 - x1) / img_w, (y2 - y1) / img_h
            lines.append(f"{TARGET_ID[target]} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}")

        shutil.copy(src_img, OUT_ROOT / "images" / split / src_img.name)
        (OUT_ROOT / "labels" / split / (src_img.stem + ".txt")).write_text("\n".join(lines))
        kept += 1
    print(f"{split}: {kept} images converted")


convert_split("train", DATASET_DIR / "UVH-26-Train" / "UVH-26-MV-Train.json", DATASET_DIR / "UVH-26-Train")
convert_split("val", DATASET_DIR / "UVH-26-Val" / "UVH-26-MV-Val.json", DATASET_DIR / "UVH-26-Val")

In [ ]:
yaml_content = f"""path: {OUT_ROOT}
train: images/train
val: images/val
names:
  0: car
  1: bike
  2: bus
  3: truck
"""
(OUT_ROOT / "data.yaml").write_text(yaml_content)
print(yaml_content)

## Train yolov8n on the converted dataset

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")
model.train(
    data=str(OUT_ROOT / "data.yaml"),
    epochs=50,
    imgsz=640,
    batch=16,
    patience=10,
    project="uvh26_runs",
    name="yolov8n_uvh26",
)

## Export and download the trained weights

In [ ]:
import shutil
from google.colab import files

best_pt = Path("uvh26_runs/yolov8n_uvh26/weights/best.pt")
shutil.copy(best_pt, "yolov8n_uvh26.pt")
files.download("yolov8n_uvh26.pt")

## Next steps — back on the laptop (not in Colab)

1. Put `yolov8n_uvh26.pt` in `backend/data/models/`.
2. In `backend/config.py`:
   - `YOLO_MODEL_NAME = "yolov8n_uvh26.pt"`
   - `VEHICLE_CLASS_IDS = [0, 1, 2, 3]`
   - `CLASS_CONFIDENCE_OVERRIDES = {1: 0.25}`  (bike moved from COCO id 3 to id 1)
3. In `backend/detection/detector.py`:
   - `CLASS_NAMES = {0: "car", 1: "bike", 2: "bus", 3: "truck"}`
4. Re-run `run_detection.py` on real lane footage and compare side-by-side
   against the stock-COCO baseline before relying on it — see
   `CLAUDE.md` section 8a.